# Phase 2d - DiCE: Diverse Counterfactual Explanations

**DOLPH-X Pipeline | Master 2 MIAGE**

**Description:** Generates diverse counterfactual examples for each instance. For a student predicted as E-P-, DiCE finds minimal feature changes that would shift the prediction towards E+P+.

| Parameter | Value | Justification |
|---|---|---|
| Method | `genetic` | More faithful to real data than `random`; does not require dimensionality reduction unlike `kdtree` |
| Models | RF, XGBoost, Decision Tree | Consistent with SHAP and LIME |
| Target class | E+P+ (encoded 0) | Ideal profile to reach |
| Counterfactuals per instance | 3 | Sufficient diversity |
| Fixed features | Course context (name, year, public type) | Not modifiable by the student |
| Variable features | Behavioural features (submissions, activity) | Real actionable levers |

**Note:** DiCE failed to generate valid counterfactuals for most instances in this experiment due to the large feature-space distance between profiles and StandardScaler normalisation constraints. The `dice_explanations.csv` output is empty. This is documented as a known limitation.

**Input:** `evaluator.pkl`

**Output:** `outputs/dice_explanations.csv` (empty - see limitation note above)

## 1. Imports

In [3]:
from MLEvaluator import MLEvaluator
from XAIEvaluator import XAIEvaluator
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import dice_ml
from dice_ml import Dice
warnings.filterwarnings("ignore")

## 2. Load Evaluator

In [5]:
evaluator = MLEvaluator.load("evaluator.pkl")

Evaluator loaded from 'evaluator.pkl'
  Available models: ['SVM', 'Logistic_Regression', 'Decision_Tree', 'KNN', 'Naive_Bayes', 'Random_Forest', 'Extra_Trees', 'Bagging', 'Gradient_Boosting', 'MLP', 'XGBoost']


## 3. DiCEEvaluator Class

In [7]:

class DiCEEvaluator(XAIEvaluator):

    def __init__(self, ml_evaluator):
        # Inherit attributes from MLEvaluator
        super().__init__(ml_evaluator)
        
        # DiCE-specific attributes
        self.dice_models     = {}
        self.counterfactuals = {}

        # Clip test values to training range
        self.X_test_clipped = self.X_test.copy()
        for col in self.feature_names:
            col_min = self.X_train[col].min()
            col_max = self.X_train[col].max()
            self.X_test_clipped[col] = self.X_test_clipped[col].clip(col_min, col_max)


        # --------------------------------------------------------
        # Fixed vs variable features (DiCE needs to know which features it can modify)
        # --------------------------------------------------------

        # Fixed feature patterns
        fixed_patterns = [
            "Unnamed",
            "PUBLIC_",           # One-Hot encoded PUBLIC
            "ACADEMIC_Y_",       # One-Hot encoded ACADEMIC_Y
            "COURSE_NAME_",      # One-Hot encoded COURSE_NAME
            "TOTAL_STUDENTS",    # number of students
            "FIRST_",            # first submission timestamps
            "LAST_",             # last submission timestamps
            "RANK_",             # rankings
            "DIFF_RANKING_",     # ranking differences
        ]

        # Identify fixed features among all columns
        self.fixed_features = [
            col for col in self.feature_names
            if any(col.startswith(p) or col == p for p in fixed_patterns)
        ]

        # Variable features: all remaining
        self.variable_features = [
            col for col in self.feature_names
            if col not in self.fixed_features
        ]

        print("DiCEEvaluator initialised")
        print(f"Available models: {list(self.trained_models.keys())}")
        print(f"Total features: {len(self.feature_names)}")
        print(f"Fixed features: {len(self.fixed_features)}")
        print(f"Variable features: {len(self.variable_features)}")

    def setup_dice(self):
        print("\n SETTING UP DiCE")
        print("=" * 60)

        # --------------------------------------------------------
        # Step 1: Préparer le dataset complet pour DiCE
        # DiCE a besoin des données X + y ensemble dans un DataFrame
        # --------------------------------------------------------
        train_data = self.X_train.copy()
        train_data["DIFFICULTY_encoded"] = self.y_train.values

        # --------------------------------------------------------
        # Step 2: Créer l'objet Data DiCE
        # Il décrit la structure du dataset :
        # - quelles colonnes sont des features
        # - quelle colonne est la cible
        # - quelles features sont continues vs catégorielles
        # --------------------------------------------------------
        dice_data = dice_ml.Data(
            dataframe=train_data,
            continuous_features=self.feature_names,  # features que DiCE peut modifier
            outcome_name="DIFFICULTY_encoded"             # colonne cible
        )

        print(f"DiCE Data object created")
        print(f"Variable features: {len(self.variable_features)}")

        # --------------------------------------------------------
        # Step 3: Créer un objet DiCE par modèle
        # --------------------------------------------------------

        # Modèles sélectionnés pour DiCE
        self.selected_models = ["Random_Forest", "XGBoost", "Decision_Tree"]

        for model_name, model in self.trained_models.items():
            if model_name not in self.selected_models:
                print(f"{model_name} skipped")
                continue

            print(f"\n   Préparation DiCE pour {model_name}...")

            try:
                # Encapsuler le modèle sklearn dans un objet DiCE
                dice_model = dice_ml.Model(
                    model=model,
                    backend="sklearn",   # on utilise sklearn
                    model_type="classifier"
                )

                # Créer l'objet Dice qui combine Data + Model
                dice_exp = Dice(
                    dice_data,
                    dice_model,
                    method="genetic"      # choix de genetic au lieu de kdtree , random
                )

                # Sauvegarder
                self.dice_models[model_name] = {
                    "dice_exp"  : dice_exp,
                    "dice_data" : dice_data,
                    "dice_model": dice_model
                }

                print(f"DiCE ready for {model_name}")

            except Exception as e:
                print(f"Error for {model_name}: {e}")
                continue

        print(f"\n DiCE préparé pour {len(self.dice_models)} modèles")
    

    def generate_counterfactuals(self, total_cfs=3):
        print("\n GENERATING COUNTERFACTUALS")
        print("=" * 60)

        for model_name, dice_objects in self.dice_models.items():
            print(f"\n Modèle : {model_name}")
            print("-" * 40)

            dice_exp = dice_objects["dice_exp"]
            model_counterfactuals = {}

            for class_name, sample_indices in self.selected_samples.items():
                if not sample_indices:
                    print(f"    No samples for {class_name}")
                    continue

                print(f"\n   Classe : {class_name}")
                class_counterfactuals = []

                for sample_idx in sample_indices:
                    # Récupérer l'échantillon
                    #query_instance = self.X_test.iloc[[sample_idx]] avant test
                    query_instance = self.X_test_clipped.iloc[[sample_idx]]

                    # Filtre : on n'explique que les prédictions correctes
                    predicted  = int(self.trained_models[model_name].predict(
                                     self.X_test_clipped.iloc[[sample_idx]])[0])
                    true_label = int(self.y_test.iloc[sample_idx])
                    if predicted != true_label:
                        print(f'      WARNINGPrédiction incorrecte -> instance {sample_idx} ignorée')
                        continue

                    # Récupérer la classe actuelle de l'élève
                    current_class = int(self.y_test.iloc[sample_idx])
                    current_class_name = self.class_mapping.get(current_class, f"Classe {current_class}")

                    # E+P+ est encodé comme 1 -> c'est notre classe cible
                    target_class = 0  # E+P+

                    # Si l'élève est déjà E+P+ -> pas besoin de contrefactuel
                    if current_class == target_class:
                        print(f"      Échantillon {sample_idx} déjà E+P+ -> ignoré")
                        continue

                    try:
                        cf_result = dice_exp.generate_counterfactuals(
                            query_instance,
                            total_CFs=total_cfs,
                            desired_class=target_class,
                            features_to_vary=self.variable_features,
                            proximity_weight=0.5,
                            diversity_weight=1.0,
                        )
                        # Vérifier que des contrefactuels ont bien été trouvés
                        cfs_df = cf_result.cf_examples_list[0].final_cfs_df
                        if cfs_df is None or len(cfs_df) == 0:
                            raise ValueError("Aucun contrefactuel trouvé")
                    
                    except Exception:
                        # Fallback: classe opposée, 1 seul contrefactuel, contraintes assouplies
                        print(f'      WARNING Fallback pour instance {sample_idx}...')
                        try:
                            cf_result = dice_exp.generate_counterfactuals(
                                query_instance,
                                total_CFs=1,
                                desired_class="opposite",
                                features_to_vary=self.variable_features,
                                proximity_weight=0.2,
                                diversity_weight=0.5,
                            )
                        except Exception as e2:
                            print(f'      ERROR Instance {sample_idx} ignorée : {e2}')
                            continue

                        # Extraire le DataFrame des contrefactuels
                        cf_df = cf_result.cf_examples_list[0].final_cfs_df

                        # Ajouter les infos de l'élève original
                        original_class = self.class_mapping.get(
                            int(self.y_test.iloc[sample_idx]),
                            f"Classe {self.y_test.iloc[sample_idx]}"
                        )

                        class_counterfactuals.append({
                            "sample_idx"    : sample_idx,
                            "original_class": original_class,
                            "query_instance": query_instance,
                            "counterfactuals": cf_df
                        })

                        print(f'Instance {sample_idx}: counterfactuals generated')

                    except Exception as e:
                        print(f"Erreur échantillon {sample_idx} : {e}")
                        continue

                model_counterfactuals[class_name] = class_counterfactuals

            self.counterfactuals[model_name] = model_counterfactuals

        print(f"\n Contrefactuels générés pour {len(self.counterfactuals)} modèles")

    def plot_counterfactuals(self, model_name):

        if model_name not in self.counterfactuals:
            print(f" No counterfactuals for {model_name}")
            return

        print(f"\n DiCE VISUALISATION: {model_name}")
        print("=" * 60)

        for class_name, class_cfs in self.counterfactuals[model_name].items():
            if not class_cfs:
                continue

            print(f"\n   Classe : {class_name}")

            for cf_data in class_cfs:
                sample_idx     = cf_data["sample_idx"]
                original_class = cf_data["original_class"]
                query_instance = cf_data["query_instance"]
                cf_df          = cf_data["counterfactuals"]

                print(f"\n   Échantillon {sample_idx}: Original class: {original_class}")

                # --------------------------------------------------------
                # GRAPHIQUE 1: Comparison table
                # --------------------------------------------------------
                try:
                    original_values = query_instance[self.variable_features].iloc[0]

                    # Identifier les features qui ont changé
                    changed_features = []
                    for feat in self.variable_features:
                        if feat in cf_df.columns:
                            # Vérifier si au moins un contrefactuel
                            # a une valeur différente de the original
                            if any(cf_df[feat] != original_values[feat]):
                                changed_features.append(feat)

                    if not changed_features:
                        print(f" No modified features detected")
                        continue

                    print(f'   Modified features: {len(changed_features)}')

                    comparison_data = {"Original": original_values[changed_features]}

                    for i, (_, cf_row) in enumerate(cf_df.iterrows()):
                        # Récupérer la classe prédite du contrefactuel
                        cf_class = self.class_mapping.get(
                            int(cf_row["DIFFICULTY_encoded"]),
                            f"Classe {int(cf_row['DIFFICULTY_encoded'])}"
                        )
                        comparison_data[f"CF{i+1} ({cf_class})"] = cf_row[changed_features]

                    comparison_df = pd.DataFrame(comparison_data)


                    print('   Comparison table:')
                    print(comparison_df.round(2).to_string())

                    # --------------------------------------------------------
                    # GRAPHIQUE 2: Bar chart of changes
                    # --------------------------------------------------------
                    n_cfs = len(cf_df)
                    fig, axes = plt.subplots(
                        1, n_cfs,
                        figsize=(6 * n_cfs, max(6, len(changed_features) * 0.4))
                    )

        
                    if n_cfs == 1:
                        axes = [axes]

                    colors = sns.color_palette("Set2", n_cfs)

                    for i, (_, cf_row) in enumerate(cf_df.iterrows()):
                        ax = axes[i]

                        cf_class = self.class_mapping.get(
                            int(cf_row["DIFFICULTY_encoded"]),
                            f"Classe {int(cf_row['DIFFICULTY_encoded'])}"
                        )

       
                        original_vals = original_values[changed_features].values
                        cf_vals       = cf_row[changed_features].values
                        differences   = cf_vals - original_vals


                        bar_colors = [
                            "#2ecc71" if d > 0  # vert = augmentation
                            else "#e74c3c"       # rouge = diminution
                            for d in differences
                        ]

                        bars = ax.barh(
                            changed_features,
                            differences,
                            color=bar_colors,
                            alpha=0.8
                        )

                        # Ligne verticale à 0
                        ax.axvline(x=0, color="black", linewidth=0.8)

                        ax.set_title(
                            f"Counterfactual {i+1}\n"
                            f"{original_class} -> {cf_class}",
                            fontsize=11,
                            fontweight="bold"
                        )
                        ax.set_xlabel("Changement par rapport à the original")
                        ax.grid(True, alpha=0.3)

                        # Ajouter les valeurs sur les barres
                        for bar, diff, orig, cf_val in zip(
                            bars, differences, original_vals, cf_vals
                        ):
                            ax.text(
                                bar.get_width() + (max(abs(differences)) * 0.02),
                                bar.get_y() + bar.get_height() / 2,
                                f"{orig:.1f} -> {cf_val:.1f}",
                                va="center",
                                fontsize=8
                            )

                    plt.suptitle(
                        f"{model_name}: Échantillon {sample_idx}\n"
                        f"Original class: {original_class}",
                        fontsize=13,
                        fontweight="bold"
                    )
                    plt.tight_layout()
                    plt.show()

                except Exception as e:
                    print(f'Visualisation error: {e}')
                    continue

    def print_summary(self):
        print("\n" + "=" * 70)
        print(" DiCE COUNTERFACTUALS SUMMARY")
        print("=" * 70)

        for model_name, model_cfs in self.counterfactuals.items():
            print(f"\n {model_name.upper().replace('_', ' ')}")
            print("-" * 60)

            total_cfs = sum(
                len(class_cfs)
                for class_cfs in model_cfs.values()
            )
            print(f"   Total instances explained : {total_cfs}")

            all_changed_features = {}

            for class_name, class_cfs in model_cfs.items():
                print(f"\n   Classe {class_name} :")

                for cf_data in class_cfs:
                    query_instance = cf_data["query_instance"]
                    cf_df          = cf_data["counterfactuals"]
                    original_class = cf_data["original_class"]

                    original_values = query_instance[
                        self.variable_features
                    ].iloc[0]

                    for feat in self.variable_features:
                        if feat in cf_df.columns:
                            if any(cf_df[feat] != original_values[feat]):
                                all_changed_features[feat] = (
                                    all_changed_features.get(feat, 0) + 1
                                )

                    if "DIFFICULTY_encoded" in cf_df.columns:
                        target_classes = [
                            self.class_mapping.get(
                                int(row["DIFFICULTY_encoded"]),
                                f"Classe {int(row['DIFFICULTY_encoded'])}"
                            )
                            for _, row in cf_df.iterrows()
                        ]
                        print(
                            f"      Échantillon {cf_data['sample_idx']} "
                            f"({original_class}) -> {target_classes}"
                        )


            if all_changed_features:
                top_features = sorted(
                    all_changed_features.items(),
                    key=lambda x: x[1],
                    reverse=True
                )[:5]

                print('   Top 5 most modified features:')
                for feat, count in top_features:
                    print(f'      {feat:40}: modified {count} times')

    def export(self, filename='outputs/dice_explanations.csv'):
        import os
        os.makedirs('outputs', exist_ok=True)
        rows = []
    
        for model_name, model_cfs in self.counterfactuals.items():
            for class_name, class_cfs in model_cfs.items():
                for cf_data in class_cfs:
                    sample_idx      = cf_data['sample_idx']
                    original_class  = cf_data['original_class']
                    query_instance  = cf_data['query_instance']
                    cf_df           = cf_data['counterfactuals']
                    original_values = query_instance[self.variable_features].iloc[0]
    
                    for cf_idx, (_, cf_row) in enumerate(cf_df.iterrows()):
                        cf_class = self.class_mapping.get(
                            int(cf_row['DIFFICULTY_encoded']), '?'
                        )
                        base = {
                            'model'          : model_name,
                            'instance_id'    : sample_idx,
                            'profil_reel'    : class_name,
                            'vrai_label'     : original_class,
                            'cf_index'       : cf_idx + 1,
                            'cf_classe_cible': cf_class,
                            'methode'        : 'DiCE',
                        }
                        for feat in self.variable_features:
                            if feat in cf_df.columns:
                                orig_val = float(original_values[feat])
                                cf_val   = float(cf_row[feat])
                                delta    = cf_val - orig_val
                                if abs(delta) > 1e-6:
                                    row = base.copy()
                                    row['feature']        = feat
                                    row['valeur_origine'] = round(orig_val, 4)
                                    row['valeur_cf']      = round(cf_val, 4)
                                    row['delta']          = round(delta, 4)
                                    row['abs_delta']      = round(abs(delta), 4)
                                    rows.append(row)
    
        df = pd.DataFrame(rows) if rows else pd.DataFrame()
        df.to_csv(filename, index=False)
        print(f'OK Export : {filename}')
        if len(rows) > 0:
            print(f'  {len(rows)} rows | {df["instance_id"].nunique()} instances')
        return df

    # MÉTHODE PRINCIPALE
    def run(self):
        print(" STARTING DiCE PIPELINE")
        print("=" * 70)

        # Step 1
        self.select_samples()

        # Step 2
        self.setup_dice()

        # Step 3
        self.generate_counterfactuals(total_cfs=3)

        # Step 4
        print("\n VISUALISING COUNTERFACTUALS")
        print("=" * 70)
        for model_name in self.counterfactuals.keys():
            self.plot_counterfactuals(model_name)

        # Step 5
        self.print_summary()

        # Step 6
        self.export()
        
        print("\n DiCE PIPELINE COMPLETE")

## 4. Execution

In [9]:
dice_eval = DiCEEvaluator(evaluator)
dice_eval.run()

DiCEEvaluator initialised
Available models: ['Decision_Tree', 'Random_Forest', 'XGBoost']
Total features: 170
Fixed features: 144
Variable features: 26
 STARTING DiCE PIPELINE

SELECTING REPRESENTATIVE INSTANCES
Class mapping: {0: 'E+P+', 1: 'E+P-', 2: 'E-P+', 3: 'E-P-'}

E+P+: 6 consensus / 7 total
  -> 4 instances selected

E+P-: 2 consensus / 4 total
  -> 2 instance(s) available (minority profile)

E-P+: 9 consensus / 9 total
  -> 4 instances selected

E-P-: 3 consensus / 6 total
  -> 3 instance(s) available (minority profile)

Total selected: 13 instances

 SETTING UP DiCE
DiCE Data object created
Variable features: 26

   Préparation DiCE pour Decision_Tree...
DiCE ready for Decision_Tree

   Préparation DiCE pour Random_Forest...
DiCE ready for Random_Forest

   Préparation DiCE pour XGBoost...
DiCE ready for XGBoost

 DiCE préparé pour 3 modèles

 GENERATING COUNTERFACTUALS

 Modèle : Decision_Tree
----------------------------------------

   Classe : E+P+
      Échantillon 5 dé

100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  1.72it/s]



   Classe : E-P+
      WARNINGPrédiction incorrecte -> instance 15 ignorée
      WARNINGPrédiction incorrecte -> instance 1 ignorée


100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  1.97it/s]



   Classe : E-P-


...

      WARNING Fallback pour instance 0...


  0%|                                                     | 0/1 [00:00<?, ?it/s]


      ERROR Instance 0 ignorée : Desired class cannot be opposite if the number of classes is more than 2.

   Classe : E-P-


100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  1.63it/s]


 Contrefactuels générés pour 3 modèles

 VISUALISING COUNTERFACTUALS

 DiCE VISUALISATION: Decision_Tree

 DiCE VISUALISATION: Random_Forest

 DiCE VISUALISATION: XGBoost

 DiCE COUNTERFACTUALS SUMMARY

 DECISION TREE
------------------------------------------------------------
   Total instances explained : 0

   Classe E+P+ :

   Classe E+P- :

   Classe E-P+ :

   Classe E-P- :

 RANDOM FOREST
------------------------------------------------------------
   Total instances explained : 0

   Classe E+P+ :

   Classe E+P- :

   Classe E-P+ :

   Classe E-P- :

 XGBOOST
------------------------------------------------------------
   Total instances explained : 0

   Classe E+P+ :

   Classe E+P- :

   Classe E-P+ :

   Classe E-P- :
OK Export : outputs/dice_explanations.csv

 DiCE PIPELINE COMPLETE
